# 04b – Modeling v2 (gleiche Pipeline wie 04)

**Lokal oder Colab** — gleiche Zellen wie `04_modeling.ipynb`, nur v2-Daten aus **03b**.

| | v1 (`04`) | v2 (`04b`) |
|---|-----------|------------|
| Input | `train_labeled.parquet`, `test_features.parquet` | `train_labeled_v2.parquet`, `test_features_v2.parquet` |
| Features | `features.py` | [features_v2.py](../scripts/features_v2.py) |
| Output | `submission_{mode}.csv` | `submission_{mode}_v2.csv` |

Voraussetzung: Notebook **03b**. **Pipeline = wie 04** (Holdout, Baselines, Submission). **Modell-Upgrade in 04b:**

- mehr Features (Score-Lags, Region-Stats, Test-91d)
- stärkeres LightGBM (mehr Bäume, L2/L1, Early Stopping 50)
- 5 Wochen-Modelle sequentiell (wie 04, Colab-stabil)
- Submission-Blend mit Persistence (35 %, Validierung wird mit ausgegeben)

Wochen-Aggregation wie in 04 → `docs/09_WEEKLY_MODELING.md`.

In [4]:
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

RANDOM_STATE = 42
VAL_REGION_FRAC = 0.2
BLEND_PERSIST = 0  # 0 = nur Modell; höher = näher an Persistence-Baseline
ES_ROUNDS = 50

# Stärker als 04 (600 trees, lr 0.05, kein L1/L2)
LGB_PARAMS = dict(
    objective="regression",
    metric="mae",
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=63,
    min_child_samples=60,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    verbose=-1,
)

for _p in (Path.cwd(), Path.cwd().parent):
    _r = _p.resolve()
    if (_r / "scripts" / "notebook_init.py").exists() and str(_r) not in sys.path:
        sys.path.insert(0, str(_r))
        break

from scripts.notebook_init import setup
from scripts.project_env import load_script

env = setup()
PROJECT_ROOT = env["project_root"]
DATA_DIR = env["data_dir"]
PROCESSED_DIR = env["processed_dir"]
SUBMISSION_DIR = env["submission_dir"]
MODE = env["mode"]

_fv2 = load_script("features_v2", PROJECT_ROOT / "scripts" / "features_v2.py")
feature_columns_v2 = _fv2.feature_columns_v2
_wm = load_script("weekly_model", PROJECT_ROOT / "scripts" / "weekly_model.py")

train_df = pd.read_parquet(PROCESSED_DIR / "train_labeled_v2.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")
FEATURES = [c for c in feature_columns_v2() if c in train_df.columns]
CAT_FEATS = ["region_id"] if "region_id" in FEATURES else None
train_df = _wm.slim_for_modeling(train_df, FEATURES)
wk = _wm.weekly_summary(train_df)
print(
    f"Train v2: {wk['daily_rows']:,} Tage → {wk['weekly_rows']:,} Wochen "
    f"(~{wk['median_weeks_per_region']:.0f}/Region) | {len(FEATURES)} Features"
)

SAMPLE_SUB_PATH = _wm.find_sample_submission(PROJECT_ROOT, DATA_DIR)
if SAMPLE_SUB_PATH is not None:
    SUBMISSION_TEMPLATE = SAMPLE_SUB_PATH
    print(f"Sample submission: {SAMPLE_SUB_PATH}")
else:
    SUBMISSION_TEMPLATE = _wm.build_submission_template(test_df["region_id"])
    print(
        "sample_submission.csv nicht gefunden — Template aus test_features "
        f"({len(SUBMISSION_TEMPLATE):,} Regionen)."
    )

print(f"Train labeled: {len(train_df):,} | Test rows: {len(test_df):,}")

Mounted at /content/drive
git pull OK → /content/DataMining_Final-Project
Umgebung: Colab | Modus: full
  Code:    /content/DataMining_Final-Project
  Daten:   /content/drive/MyDrive/DataMining/DataMining_Final-Project/data
  Train:   /content/drive/MyDrive/DataMining/DataMining_Final-Project/data/train.csv (OK)
  Test:    /content/drive/MyDrive/DataMining/DataMining_Final-Project/data/test.csv (OK)
  Outputs: /content/drive/MyDrive/DataMining/DataMining_Final-Project/outputs
Train v2: 1,757,936 Tage → 1,726,554 Wochen (~782/Region) | 76 Features
Sample submission: /content/DataMining_Final-Project/resources/sample_submission.csv
Train labeled: 1,757,936 | Test rows: 204,568


## 1. Train/Valid (wöchentliche Fenster → 5 Wochen Targets)

In [5]:
X_tr, y_tr, X_va, y_va, val_regions = _wm.build_region_holdout(
    train_df, FEATURES, val_region_frac=VAL_REGION_FRAC, seed=RANDOM_STATE
)
print(f"Train-Fenster (wöchentlich): {len(X_tr):,} | Val-Regionen: {len(val_regions)}")
print(f"y shape: train {y_tr.shape}, val {y_va.shape}  (~{len(X_tr)/wk['regions']:.0f} Fenster/Region)")

Train-Fenster (wöchentlich): 1,373,153 | Val-Regionen: 449
y shape: train (1373153, 5), val (449, 5)  (~611 Fenster/Region)


## 2. Baselines (MAE wie Kaggle)

In [6]:
train_weekly = _wm.daily_to_weekly(train_df)
last_score = train_weekly.sort_values("ordinal").groupby("region_id")["score"].last()
region_median = train_weekly.groupby("region_id")["score"].median()

def eval_preds(y_true, y_pred, name):
    print(f"{name:32s}  MAE={_wm.mae_kaggle(y_true, y_pred):.4f}")


persist = np.column_stack([
    last_score.reindex(val_regions).fillna(0).to_numpy() for _ in range(5)
])
eval_preds(y_va, persist, "Persistence (letzter Score ×5)")

med = np.column_stack([
    region_median.reindex(val_regions).fillna(train_df["score"].median()).to_numpy()
    for _ in range(5)
])
eval_preds(y_va, med, "Regional median ×5")

Persistence (letzter Score ×5)    MAE=0.0321
Regional median ×5                MAE=0.4682


## 3. LightGBM v2 (wie 04: 5 Wochen sequentiell, stärkere Params)

In [7]:
def y_week(y, w):
    """LightGBM 4.x: 1D-Labels als list (kein y[:, w]-Slice, kein 2D-y_va in eval_set)."""
    return np.asarray(y, dtype=np.float64)[:, w].ravel().tolist()


models = []
val_preds = np.zeros_like(y_va)

for week in range(5):
    p = dict(LGB_PARAMS)
    p["random_state"] = RANDOM_STATE + week
    p["n_jobs"] = -1
    m = lgb.LGBMRegressor(**p)
    fit_kw = dict(
        eval_set=[(X_va, y_week(y_va, week))],
        eval_metric="mae",
        callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)],
    )
    if CAT_FEATS:
        fit_kw["categorical_feature"] = CAT_FEATS
    m.fit(X_tr, y_week(y_tr, week), **fit_kw)
    models.append(m)
    val_preds[:, week] = _wm.clip_scores(m.predict(X_va))

eval_preds(y_va, val_preds, "LightGBM v2 (5 Modelle)")

if BLEND_PERSIST > 0:
    blend_val = _wm.clip_scores(
        (1 - BLEND_PERSIST) * val_preds + BLEND_PERSIST * persist
    )
    eval_preds(
        y_va,
        blend_val,
        f"Blend valid ({(1 - BLEND_PERSIST):.0%} Modell + {BLEND_PERSIST:.0%} Persist)",
    )

LightGBM v2 (5 Modelle)           MAE=0.1423


## 4. Finales Training, Persistence-Blend & Submission

In [ ]:
X_all, y_all, _ = _wm.build_sliding_samples(
    train_df, FEATURES, already_weekly=False
)  # weekly collapse inside (same as 04)

# Finales Training: best_iteration pro Woche aus Validierung (wie 04, aber v2-Params)
final_models = []
for week in range(5):
    p = dict(LGB_PARAMS)
    p["n_estimators"] = getattr(models[week], "best_iteration_", None) or LGB_PARAMS["n_estimators"]
    p["random_state"] = RANDOM_STATE + week
    p["n_jobs"] = -1
    m = lgb.LGBMRegressor(**p)
    fit_kw = {}
    if CAT_FEATS:
        fit_kw["categorical_feature"] = CAT_FEATS
    m.fit(X_all, y_week(y_all, week), **fit_kw)
    final_models.append(m)
print("Finales Training: 5 Modelle (best_iteration aus Validierung)")

X_test = _wm.test_features_last_row(test_df, FEATURES)
test_preds = _wm.predict_week_columns(final_models, X_test)

if BLEND_PERSIST > 0:
    last_s = last_score.reindex(X_test["region_id"]).fillna(0).to_numpy(dtype=float)
    persist_test = np.column_stack([last_s for _ in range(5)])
    test_preds = _wm.clip_scores(
        (1 - BLEND_PERSIST) * test_preds + BLEND_PERSIST * persist_test
    )
    print(f"Submission-Blend: {(1 - BLEND_PERSIST):.0%} Modell + {BLEND_PERSIST:.0%} Persistence")

submission = _wm.submission_frame(X_test["region_id"], test_preds)
submission = _wm.align_to_sample_submission(submission, SUBMISSION_TEMPLATE)

out_path = SUBMISSION_DIR / f"submission_{MODE}_v2.csv"
submission.to_csv(out_path, index=False)

print(f"Gespeichert: {out_path}")
print(f"Zeilen: {len(submission):,} (erwartet 2.248)")
print(f"Spalten: {list(submission.columns)}")
submission.head(10)